# Transformer solver — ranking, features, and attention

The transformer is not a move generator. It is a **placement scorer**: every
legal reachable candidate is tokenized (board cells + piece + position +
surface + hole-fit), run through a small encoder, and given a quality score.
Play = `argmax` over that list.

That one change — *evaluate* instead of *emit* — is why this model clears
**150+ lines** while VAE/GAN stall in the single digits.

This notebook is aimed at an AI researcher: feature design, the scoring heads,
and **whether attention goes to the right cells and tokens**.

| | |
|---|---|
| **Idea** | Contrastive scoring of legal candidates |
| **Play** | Enumerate BFS placements, pick highest `score` |
| **Shipped config** | `d_model=64`, 4 heads, 2 layers (not the CLI defaults of 128/8/4) |

Weights: `models/transformer_scorer/transformer_scorer_best.keras`.
Train: `uv run train-transformer`.


## 0. Setup


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

nb_dir = Path.cwd()
if not (nb_dir / "nb_setup.py").exists():
    nb_dir = nb_dir / "notebooks"
sys.path.insert(0, str(nb_dir))

import nb_setup as N
N.setup_notebook()

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from puzzlegrid.pieces import PieceType
from puzzlegrid import placement
import transformer_features as tf_feat
from train_transformers import (
    PositionIndicesLayer,
    build_scorer_model,
    load_games,
    build_examples,
)
from replay_transformers import TransformerScorerPlayer

print("TF", tf.__version__)
print("FEATURE_KEYS", tf_feat.FEATURE_KEYS)


## 1. Ranking beats generation

VAE and GAN emit a `(rotation, position)` and snap it onto the legal set. The
transformer **sees every legal option** and learns `evaluate_placement`.

Training labels (from `transformer_features.candidate_labels`) are a shaped
score: line clears dominate, holes are punished, near-full rows are rewarded.
Auxiliary heads predict `lines_cleared` and `holes_created`.

Token sequence (length 204):

| index | token | meaning |
|------:|-------|---------|
| 0–199 | board cells | row-major occupancy, + positional embedding |
| 200 | piece | concat(piece one-hot, rotation one-hot) |
| 201 | position | normalized (row, col) of this candidate |
| 202 | surface | 10 column heights of the *current* stack |
| 203 | holefit | 5-d descriptor of what *this* move does |

If attention is working, board-cell keys under the piece and along the surface
should light up, and the hole-fit / surface tokens should be used — not ignored.


## 2. Feature design — look at one candidate


In [ ]:
PIECE = "I"
SCENARIO = "almost_line"
board = N.SCENARIOS[SCENARIO]()
piece_type = PieceType[PIECE]
cands = placement.get_valid_placements(board, piece_type)
print("legal candidates", len(cands))

# Two candidates: one that should clear, one that should not (if the scenario allows).
feats_all = []
labels_all = []
for (r, c, rot) in cands:
    feats_all.append(tf_feat.candidate_features(board, piece_type, r, c, rot))
    labels_all.append(tf_feat.candidate_labels(board, piece_type, r, c, rot))

scores = np.array([lab[0] for lab in labels_all])
lines = np.array([lab[1] for lab in labels_all])
holes = np.array([lab[2] for lab in labels_all])
best_i, worst_i = int(np.argmax(scores)), int(np.argmin(scores))
print("teacher-best", cands[best_i], "score", scores[best_i], "lines", lines[best_i], "holes", holes[best_i])
print("teacher-worst", cands[worst_i], "score", scores[worst_i], "lines", lines[worst_i], "holes", holes[worst_i])
print("holefit best ", feats_all[best_i]["holefit"],
      "(holes_created, holes_filled, bump_delta, max_h, clean)")
print("surface      ", np.round(feats_all[best_i]["surface"], 2))

N.show_boards(
    [
        {"board": board, "piece": cands[best_i], "piece_type": PIECE,
         "title": f"teacher-best  s={scores[best_i]:.2f}  L={lines[best_i]}"},
        {"board": board, "piece": cands[worst_i], "piece_type": PIECE,
         "title": f"teacher-worst s={scores[worst_i]:.2f}  L={lines[worst_i]}"},
    ],
    cols=2,
)


## 3. Architecture

Shipped model: **2 layers × 4 heads**, `d_model=64`, `key_dim=16`. CLI defaults
in `build_scorer_model` are larger; the 150+ line run used this smaller net.

```
cells (200×64) + piece + position + surface + holefit   →  (B, 204, 64)
  ×2  MultiHeadAttention + residual LN + FFN (256→64)
  GAP → MLP → score / lines_cleared / holes_created
```


In [ ]:
# Untrained twin with the SHIPPED sizes, for a readable summary.
tiny = build_scorer_model(d_model=64, num_heads=4, num_layers=2)
tiny.summary()


## 4. Optional short training

Building contrastive examples enumerates up to `max_candidates` legal placements
per move, so even one game is a few hundred rows. Full training:
`uv run train-transformer`.


In [ ]:
RUN_SHORT_TRAINING = False

if RUN_SHORT_TRAINING:
    games = load_games(str(N.DATA_DIR), max_files=1)
    X, y, w = build_examples(games, max_candidates=12)
    model = build_scorer_model(d_model=64, num_heads=4, num_layers=2)
    hist = model.fit(X, y, sample_weight=w, epochs=1, batch_size=32, verbose=2)
    print({k: v[-1] for k, v in hist.history.items()})
else:
    print("Skipping in-notebook training; loading shipped scorer.")


## 5. Load the shipped scorer


In [ ]:
scorer = tf.keras.models.load_model(
    str(N.TRANSFORMER_PATH),
    compile=False,
    custom_objects={"PositionIndicesLayer": PositionIndicesLayer},
)
print("inputs", [t.name for t in scorer.inputs])
print("attn layers", [l.name for l in scorer.layers if "attn" in l.name])
for layer in scorer.layers:
    if isinstance(layer, tf.keras.layers.MultiHeadAttention):
        print(layer.name, "heads", layer.num_heads, "key_dim", layer.key_dim)

TOKEN_SLICE = {
    "cells": slice(0, 200),
    "piece": 200,
    "position": 201,
    "surface": 202,
    "holefit": 203,
}
N_TOKENS = 204


## 6. Score every legal candidate

Compare the **model score** to the **teacher label** and to simulated line
clears. Agreement on the argmax is what matters at play time.


In [ ]:
def batch_from_board(board, piece_name):
    piece = PieceType[piece_name]
    cands = placement.get_valid_placements(board, piece)
    batch = {k: [] for k in tf_feat.FEATURE_KEYS}
    teacher = []
    sim_lines = []
    for (r, c, rot) in cands:
        fd = tf_feat.candidate_features(board, piece, r, c, rot)
        for k in tf_feat.FEATURE_KEYS:
            batch[k].append(fd[k])
        teacher.append(tf_feat.candidate_labels(board, piece, r, c, rot))
        _, lc = placement.simulate_placement(board, piece, r, c, rot)
        sim_lines.append(lc)
    X = {k: np.array(v) for k, v in batch.items()}
    return cands, X, teacher, np.array(sim_lines)

cands, X, teacher, sim_lines = batch_from_board(board, PIECE)
pred_score, pred_lines, pred_holes = scorer.predict(X, verbose=0)
pred_score = pred_score.reshape(-1)
pred_lines = pred_lines.reshape(-1)
pred_holes = pred_holes.reshape(-1)
teacher_score = np.array([t[0] for t in teacher])

order = np.argsort(-pred_score)
rows = []
for i in order:
    r, c, rot = cands[i]
    rows.append({
        "pred": float(pred_score[i]),
        "teacher": float(teacher_score[i]),
        "lines": int(sim_lines[i]),
        "pred_L": float(pred_lines[i]),
        "pred_H": float(pred_holes[i]),
        "col": c, "row": r, "rot": rot, "i": int(i),
    })
N.ranked_table(rows, ["pred", "teacher", "lines", "pred_L", "pred_H", "col", "row", "rot"], n=15)
print("model argmax", cands[int(np.argmax(pred_score))],
      "teacher argmax", cands[int(np.argmax(teacher_score))])


In [ ]:
pick_i = int(np.argmax(pred_score))
worst_i = int(np.argmin(pred_score))
heat = np.full((20, 10), np.nan)
for (r, c, rot), s in zip(cands, pred_score):
    if np.isnan(heat[r, c]) or s > heat[r, c]:
        heat[r, c] = s
N.show_boards(
    [
        {"board": board, "piece": cands[pick_i], "piece_type": PIECE,
         "title": f"model pick  score={pred_score[pick_i]:.3f}  L={sim_lines[pick_i]}"},
        {"board": board, "heatmap": heat,
         "title": "model score at resting cells"},
        {"board": board, "piece": cands[worst_i], "piece_type": PIECE,
         "title": f"model worst score={pred_score[worst_i]:.3f}"},
    ],
    cols=3,
)


## 7. Extract attention

The saved model was not compiled with `return_attention_scores=True`, but the
`MultiHeadAttention` layers still accept that flag when called directly.

Scores shape: `(batch, heads, query, key)` over 204 tokens. We look at:

1. **Board ← special tokens** — does piece / surface / holefit attend to cells?
2. **Board ← board** (mean query) — occupancy self-attention
3. **Head specialization** — 4 heads × 2 layers
4. **Best vs worst candidate** — did attention move to the gap / well?


In [ ]:
def attention_for(X_one):
    """X_one: feature dict with batch dimension 1. Returns list of (1,H,Q,K) arrays."""
    seq_model = getattr(scorer, "_seq_model", None)
    if seq_model is None:
        scorer._seq_model = tf.keras.Model(
            scorer.inputs, scorer.get_layer("concatenate_1").output
        )
        seq_model = scorer._seq_model
    x = seq_model(X_one, training=False)
    scores = []
    for i in range(2):
        attn_layer = scorer.get_layer(f"attn_{i}")
        attn_out, attn_scores = attn_layer(x, x, return_attention_scores=True, training=False)
        scores.append(attn_scores.numpy())
        x = scorer.get_layer(f"add1_{i}")([x, attn_out])
        x = scorer.get_layer(f"norm1_{i}")(x)
        ffn_out = scorer.get_layer(f"ffn_{i}")(x, training=False)
        x = scorer.get_layer(f"add2_{i}")([x, ffn_out])
        x = scorer.get_layer(f"norm2_{i}")(x)
    return scores  # [layer0, layer1], each (1, 4, 204, 204)


def one_index(X, i):
    return {k: v[i : i + 1] for k, v in X.items()}


def cells_to_board(vec200):
    return np.asarray(vec200).reshape(20, 10)


attn_pick = attention_for(one_index(X, pick_i))
attn_worst = attention_for(one_index(X, worst_i))
print("layer0", attn_pick[0].shape, "layer1", attn_pick[1].shape)


### 7a. Special-token → board (layer 0, mean over heads)

Each special token’s attention over the 200 cells, reshaped to 20×10. You want
`holefit` and `piece` to concentrate on the landing site and any gap the move
fills; `surface` should trace the top of the stack.


In [ ]:
def show_special_to_board(attn_layers, title, piece_placement=None):
    # mean heads, layer 0 and layer 1 stacked
    fig, axes = plt.subplots(2, 4, figsize=(12, 10))
    names = ["piece", "position", "surface", "holefit"]
    idxs = [200, 201, 202, 203]
    for li, attn in enumerate(attn_layers):
        # attn: (1, heads, Q, K)
        mean_heads = attn[0].mean(axis=0)  # (Q, K)
        for j, (name, qi) in enumerate(zip(names, idxs)):
            ax = axes[li, j]
            board_attn = cells_to_board(mean_heads[qi, :200])
            N.show_board(
                board,
                piece=piece_placement,
                piece_type=PIECE if piece_placement else None,
                heatmap=board_attn,
                title=f"L{li} {name}→board",
                ax=ax,
                show=False,
            )
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

show_special_to_board(attn_pick, "Attention on MODEL PICK", cands[pick_i])
show_special_to_board(attn_worst, "Attention on MODEL WORST", cands[worst_i])


### 7b. Per-head board attention (layer 1)

Layer 1 is closer to the score. Mean over queries restricted to cell tokens,
keys also cells — a spatial self-attention map per head. Heads that are “doing
something” focus on the surface, the hole, or the piece footprint; a collapsed
head is nearly uniform.


In [ ]:
def show_heads(attn, title):
    # attn (1, 4, 204, 204) — cell queries attending to cell keys
    heads = attn[0, :, :200, :200]  # (4, 200, 200)
    fig, axes = plt.subplots(1, 4, figsize=(14, 6.2))
    for h in range(4):
        # average query attention to each key cell
        key_mass = heads[h].mean(axis=0).reshape(20, 10)
        entropy = -np.sum(heads[h].mean(axis=0) * np.log(heads[h].mean(axis=0) + 1e-12))
        N.show_board(
            board,
            piece=cands[pick_i],
            piece_type=PIECE,
            heatmap=key_mass,
            title=f"head {h}  H={entropy:.2f}",
            ax=axes[h],
            show=False,
        )
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

show_heads(attn_pick[1], "Layer 1 cell→cell key-mass (model pick)")

# Special-token attention mass: how much does each head use holefit / surface?
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for li, ax in enumerate(axes):
    mass = attn_pick[li][0, :, :, 200:].mean(axis=1)  # (heads, 4 special keys)
    im = ax.imshow(mass, cmap="magma", aspect="auto")
    ax.set_xticks(range(4)); ax.set_xticklabels(["piece", "pos", "surface", "holefit"], rotation=30)
    ax.set_yticks(range(4)); ax.set_yticklabels([f"h{h}" for h in range(4)])
    ax.set_title(f"layer {li}: head → special-token key mass")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


## 8. Is attention on the right things?

Two quantitative checks a researcher can re-run on any scenario:

1. **Landing-site mass** — fraction of `piece`/`holefit` attention that lands
   on cells the candidate piece occupies (plus 4-neighborhood). Higher on the
   pick than on a random legal move is a good sign.
2. **Feature ablation** — zero `holefit` or `surface` at inference and watch
   the argmax flip. If the pick is unchanged, that token is unused.


In [ ]:
def landing_mass(attn_layers, placement, token="holefit"):
    qi = TOKEN_SLICE[token]
    cells = N.piece_cells(PIECE, *placement)
    mask = np.zeros(200, dtype=bool)
    for r, c in cells:
        if 0 <= r < 20 and 0 <= c < 10:
            mask[r * 10 + c] = True
            for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
                rr, cc = r + dr, c + dc
                if 0 <= rr < 20 and 0 <= cc < 10:
                    mask[rr * 10 + cc] = True
    # last layer, mean heads, that query → cells
    a = attn_layers[-1][0, :, qi, :200].mean(axis=0)
    return float(a[mask].sum() / (a.sum() + 1e-9))

print("holefit mass on pick footprint ", landing_mass(attn_pick, cands[pick_i], "holefit"))
print("holefit mass on worst footprint", landing_mass(attn_worst, cands[worst_i], "holefit"))
print("piece  mass on pick footprint  ", landing_mass(attn_pick, cands[pick_i], "piece"))
print("surface mass on pick footprint ", landing_mass(attn_pick, cands[pick_i], "surface"))


def ablate(X, key, fill=0.0):
    Y = {k: v.copy() for k, v in X.items()}
    Y[key] = np.full_like(Y[key], fill)
    return Y

def argmax_of(Xin):
    s, _, _ = scorer.predict(Xin, verbose=0)
    return int(np.argmax(s.reshape(-1)))

base = argmax_of(X)
print("base pick", cands[base])
for key in ("holefit", "surface", "position", "piece_type", "rotation"):
    j = argmax_of(ablate(X, key, 0.0))
    print(f"ablate {key:12s} → {cands[j]}   changed={j != base}")


## 9. Researcher experiments worth running

Change `PIECE` / `SCENARIO` at the top and re-run from §2. Suggested probes:

| Probe | What you hope to see |
|---|---|
| `right_well` + `I` | holefit/piece attention down the well; pick is vertical I in col 9 |
| `almost_line` + `O`/`I` | score head and `lines_cleared` head agree on the completing move |
| `bumpy` + `T` | holefit token used; ablate `holefit` should change the pick |
| `overhang_pocket` | BFS candidates only; attention should not “want” unreachable cells |
| Real game board (`N.board_before_nth_place`) | attention on the actual surface, not the empty sky |

Attention entropy per head (printed on the head maps) is a quick collapse check:
very high entropy ≈ uniform ≈ unused head.


## 10. Solver: in-process play


In [ ]:
player = TransformerScorerPlayer(
    str(N.TRANSFORMER_PATH),
    use_inprocess=True,
    line_clear_bonus=0.0,
)
player.create_game(piece_sequence=["I", "O", "T", "S", "Z", "J", "L"] * 20)
# Drop in our scenario as the starting stack.
player._local._game.board.grid = [row[:] for row in board]
final = player.play(max_moves=40, delay=0.0)
print("lines", final.get("lines_cleared"),
      "pieces", final.get("pieces_placed"),
      "score", final.get("score"),
      "status", final.get("status"))
N.show_board(final["board"], title=f"transformer after 40 moves  lines={final.get('lines_cleared')}")


## 11. Takeaways

- The transformer **attends over a designed token mix**, not raw pixels only.
  Surface and hole-fit are inductive biases that make “paying attention to the
  right things” checkable.
- Use **special-token → board maps**, **per-head entropy**, **landing-site
  mass**, and **ablation of holefit/surface** as a small eval suite whenever
  you retrain.
- Generation (VAE/GAN/distilled policy) and ranking are not interchangeable.
  The distilled policy in `models/transformer_policy_soft/` is the control:
  same family, worse interface, ~5 lines.
- Agents (`uv run play-agents`) wrap this scorer with BFS navigation; the LLM
  is only a recovery fallback.

That is the study arc: imitate (VAE) → add a value-ish head (GAN) → evaluate
every legal move (transformer).
